In [1]:
# **MultiDevice Demonstration Workflow**


## **Overview**

The <code>MultiDevice</code> class provides a generic framework for loading, aligning, and structuring data from multiple identical Harp devices into unified <code>xr.DataArray</code> structures. It replaces the previous <code>BehaviorNosepoke</code> class with a more flexible, device-agnostic approach.

This notebook demonstrates:
1. What arguments need to be defined to use <code>MultiDevice</code>
2. How to instantiate and use <code>MultiDevice</code> directly
3. How to use the <code>Nosepoke</code> convenience subclass
4. How to use the deprecated <code>BehaviorNosepoke</code> wrapper for backwards compatibility

# **Part 1: Defining the Required Arguments**
-----

## 1.1| Imports and Paths

To use <code>MultiDevice</code>, we need to import it along with its subclasses <code>Nosepoke</code> and <code>BehaviorNosepoke</code> from <code>multidevice_mod</code>. We also need to provide:
- <code>experiment_directory_path</code>: Path to the root directory of the Bonsai experiment session
- <code>harp_device_yaml_path</code>: Path to the YAML schema file for the harp device type

In [2]:
#======= Setup and Necessary User Inputs
import sys
import pandas as pd
import numpy as np
import xarray as xr
from pathlib import Path

# project_root = Path('../../')
# sys.path.insert(0, str(project_root))

from Refactor.MultiDevices.multidevice_mod import MultiDevice, Nosepoke, BehaviorNosepoke

experiment_directory_path= './Bonsai_logs/2025-10-07T18-08-45'
harp_device_yaml_path= './device.yml'

## 1.2| Device Configuration

As described in the main Demonstration Workflow, users need to define which devices are present and which registers each device has:
- <code>device_list</code>: A list of device folder names present in the experiment directory (e.g. <code>['Behavior0', 'Behavior1']</code>)
- <code>device_registers_dict</code>: A dictionary mapping each device name to a list of register addresses that should be loaded for that device

In [3]:
#======= Device Configuration
device_list = ['Behavior0', 'Behavior1']

device_registers_dict = {
    'Behavior0': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior1': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
}

## 1.3| Channel, LocalID, and Register Configuration

As with the previous <code>BehaviorNosepoke</code> class, users need to define the relationships between unified channel names, device local IDs, and register addresses. These are the same definitions used in Part 2 of the main Demonstration Workflow:

- <code>channel_list</code>: A list of unified channel names (e.g. <code>Nosepoke0</code> ... <code>Nosepoke5</code>)
- <code>channel_type_localIDs</code>: For each data type (type_key), the ordered list of local port names that cycle across devices
- <code>channel_type_registerIDs</code>: For each data type, a dictionary mapping each local port name to its register address

In [4]:
#======= Channel Names
channel_list = ['Nosepoke0',
                'Nosepoke1',
                'Nosepoke2',
                'Nosepoke3',
                'Nosepoke4',
                'Nosepoke5',
]

#======= Per-type Local IDs
Nosepoke_Activations_localIDs = ['DIPort0',
                                 'DIPort1',
                                 'DIPort2',
]

Nosepoke_LED_Activations_localIDs = ['DOPort0',
                                     'DOPort1',
                                     'DOPort2',
]

Nosepoke_Valve_Activations_localIDs = ['SupplyPort0',
                                       'SupplyPort1',
                                       'SupplyPort2',
]

Nosepoke_Reward_Release_localIDs = ['PulseSupplyPort0',
                                    'PulseSupplyPort1',
                                    'PulseSupplyPort2',
]

channel_type_localIDs = {
    'Activations': Nosepoke_Activations_localIDs,
    'LED_Activations': Nosepoke_LED_Activations_localIDs,
    'Valve_Activations': Nosepoke_Valve_Activations_localIDs,
    'Reward_Release': Nosepoke_Reward_Release_localIDs,
}

#======= Per-type Register Addresses
Activations_localID_register_dict = {
    'DIPort0': '32',
    'DIPort1': '32',
    'DIPort2': '32',
}
LED_Activations_localID_register_dict = {
    'DOPort0': '34',
    'DOPort1': '34',
    'DOPort2': '34',
}

Valve_Activations_localID_register_dict = {
    'SupplyPort0': '34',
    'SupplyPort1': '34',
    'SupplyPort2': '34',
}

Reward_Release_localID_register_dict = {
    'PulseSupplyPort0': '49',
    'PulseSupplyPort1': '50',
    'PulseSupplyPort2': '51',
}

channel_type_registerIDs = {
    'Activations': Activations_localID_register_dict,
    'LED_Activations': LED_Activations_localID_register_dict,
    'Valve_Activations': Valve_Activations_localID_register_dict,
    'Reward_Release': Reward_Release_localID_register_dict,
}

type_keys = list(channel_type_localIDs.keys())

## 1.4| Building Virtual Maps

<code>MultiDevice</code> uses a new data structure called a **virtual map** to define how unified channel names map to physical hardware. A virtual map is a nested dictionary with the following structure:

```python
virtual_maps = {
    'type_key': {
        'ChannelName': {'device': 'DeviceName', 'register': 'RegisterAddress', 'localID': 'PortName'},
        ...
    },
    ...
}
```

This replaces the separate <code>channel_ref_dict</code> used by the old <code>BehaviorNosepoke</code> class. Each entry maps a unified channel name to its physical device, the register address the data lives on, and the local port name (column) within that register's DataFrame.

The mapping follows a **device-major** cycling pattern: channels are assigned to local ports within the first device, then the second device, and so on. For example, with 2 devices and 3 local ports per device:

| Channel | Device | localID |
|---|---|---|
| Nosepoke0 | Behavior0 | Port0 |
| Nosepoke1 | Behavior0 | Port1 |
| Nosepoke2 | Behavior0 | Port2 |
| Nosepoke3 | Behavior1 | Port0 |
| Nosepoke4 | Behavior1 | Port1 |
| Nosepoke5 | Behavior1 | Port2 |

In [5]:
#======= Build virtual_maps from channel config (device-major cycling)
virtual_maps = {}

for tk in type_keys:
    local_ids = channel_type_localIDs[tk]
    reg_map   = channel_type_registerIDs[tk]
    n_locals  = len(local_ids)

    vmap = {}
    for i, channel in enumerate(channel_list):
        dev_idx = i // n_locals              # which device
        lid     = local_ids[i % n_locals]    # which local port
        vmap[channel] = {
            'device':   device_list[dev_idx],
            'register': reg_map[lid],
            'localID':  lid,
        }
    virtual_maps[tk] = vmap

#======= Inspect
print(f"virtual_maps keys: {list(virtual_maps.keys())}\n")
for ch, info in virtual_maps['Activations'].items():
    print(f"  {ch:>12s}  →  device={info['device']}, register={info['register']}, localID={info['localID']}")

virtual_maps keys: ['Activations', 'LED_Activations', 'Valve_Activations', 'Reward_Release']

     Nosepoke0  →  device=Behavior0, register=32, localID=DIPort0
     Nosepoke1  →  device=Behavior0, register=32, localID=DIPort1
     Nosepoke2  →  device=Behavior0, register=32, localID=DIPort2
     Nosepoke3  →  device=Behavior1, register=32, localID=DIPort0
     Nosepoke4  →  device=Behavior1, register=32, localID=DIPort1
     Nosepoke5  →  device=Behavior1, register=32, localID=DIPort2


# **Part 2: Using MultiDevice**
-----

## 2.1| Instantiating MultiDevice

When <code>MultiDevice</code> is instantiated with both <code>experiment_directory_path</code> and <code>data_keys</code>, it automatically runs the full processing pipeline:
1. Collects DataFrames from all device folders for all registers
2. Collects and merges timestamps across devices for each data_key
3. Constructs a base <code>xr.DataArray</code> per data_key with unified channel names and a shared time axis
4. Constructs a lookup array per data_key showing the (device, register, localID) mapping
5. Updates each DataArray with the actual data values from the device DataFrames

The results are stored in:
- <code>self.data_arrays</code>: <code>{data_key: xr.DataArray}</code>
- <code>self.lookup_arrays</code>: <code>{data_key: xr.DataArray}</code>
- <code>self.lookup_virtual_coords</code>: <code>{data_key: dict}</code>

In [6]:
#======= Instantiate MultiDevice
md = MultiDevice(
    virtual_maps              = virtual_maps,
    global_coord_name         = 'peripherals',
    virtual_coord_names       = ['device', 'register', 'localID'],
    data_keys                 = type_keys,
    experiment_directory_path = experiment_directory_path,
    harp_device_yaml_path     = harp_device_yaml_path,
    device_type               = 'Behavior',
    device_list               = device_list,
    device_registers_dict     = device_registers_dict,
    verbose                   = True,
)


--- Processing devices and registers ---

Processing device: Behavior0
  - Loaded register TimestampSeconds (address 8) with shape (1313, 1)
  - Loaded register DigitalInputState (address 32) with shape (8, 4)
  - Loaded register OutputSet (address 34) with shape (424, 14)
  - Loaded register OutputClear (address 35) with shape (447, 14)
  - Loaded register AnalogData (address 44) with shape (1312286, 3)
  - Loaded register StartCameras (address 78) with shape (1, 2)
  - Loaded register Camera0Frame (address 92) with shape (78682, 1)

Processing device: Behavior1
  - Loaded register TimestampSeconds (address 8) with shape (1313, 1)
  - Loaded register DigitalInputState (address 32) with shape (6, 4)
  - Loaded register OutputSet (address 34) with shape (423, 14)
  - Loaded register OutputClear (address 35) with shape (447, 14)
  - Loaded register AnalogData (address 44) with shape (1312389, 3)
  - Loaded register StartCameras (address 78) with shape (1, 2)
  - Loaded register Camera0F

## 2.2| Inspecting the Outputs

The <code>data_arrays</code> dictionary contains one <code>xr.DataArray</code> per type_key. Each DataArray has:
- A <code>time</code> dimension containing the merged, sorted timestamps from all devices for that type_key
- A <code>peripherals</code> dimension containing the unified channel names
- Virtual coordinate arrays (<code>device</code>, <code>register</code>, <code>localID</code>) attached along the peripherals dimension

In [7]:
#======= Inspect data_arrays
print(f"data_arrays keys: {list(md.data_arrays.keys())}\n")

for dk in md.data_arrays:
    da = md.data_arrays[dk]
    print(f"\n{'='*60}")
    print(f"{dk}")
    print(f"{'='*60}")
    display(da)
    display(da.to_dataframe())

data_arrays keys: ['Activations', 'LED_Activations', 'Valve_Activations', 'Reward_Release']


Activations


<xarray.DataArray 'Activations_data' (Time: 14, peripherals: 6)> Size: 672B
array([[nan, nan, nan,  0.,  0.,  1.],
       [nan, nan, nan,  0.,  0.,  0.],
       [nan, nan, nan,  0.,  0.,  1.],
       [nan, nan, nan,  0.,  0.,  0.],
       [ 1.,  0.,  0., nan, nan, nan],
       [ 0.,  0.,  0., nan, nan, nan],
       [ 1.,  0.,  0., nan, nan, nan],
       [ 0.,  0.,  0., nan, nan, nan],
       [nan, nan, nan,  0.,  1.,  0.],
       [nan, nan, nan,  0.,  0.,  0.],
       [ 0.,  1.,  0., nan, nan, nan],
       [ 0.,  0.,  0., nan, nan, nan],
       [ 0.,  0.,  1., nan, nan, nan],
       [ 0.,  0.,  0., nan, nan, nan]])
Coordinates:
  * Time         (Time) float64 112B 1.056e+05 1.056e+05 ... 1.06e+05 1.06e+05
  * peripherals  (peripherals) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device       (peripherals) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register     (peripherals) <U2 48B '32' '32' '32' '32' '32' '32'
    localID      (peripherals) <U7 168B 'DIPort0' 'DIPort1' ... 'DIPort2'
Attributes:
    description:  Base DataArray for unified coordinates of type Activations_...
    source:       Constructed using construct_base_da function

device register  localID  Activations_data
Time          peripherals                                               
105556.648256 Nosepoke0    Behavior0       32  DIPort0               NaN
              Nosepoke1    Behavior0       32  DIPort1               NaN
              Nosepoke2    Behavior0       32  DIPort2               NaN
              Nosepoke3    Behavior1       32  DIPort0               0.0
              Nosepoke4    Behavior1       32  DIPort1               0.0
...                              ...      ...      ...               ...
106043.443744 Nosepoke1    Behavior0       32  DIPort1               0.0
              Nosepoke2    Behavior0       32  DIPort2               0.0
              Nosepoke3    Behavior1       32  DIPort0               NaN
              Nosepoke4    Behavior1       32  DIPort1               NaN
              Nosepoke5    Behavior1       32  DIPort2               NaN

[84 rows x 4 columns]


LED_Activations


<xarray.DataArray 'LED_Activations_data' (Time: 521, peripherals: 6)> Size: 25kB
array([[ 0.,  0.,  0., nan, nan, nan],
       [nan, nan, nan,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.]])
Coordinates:
  * Time         (Time) float64 4kB 1.05e+05 1.05e+05 ... 1.063e+05 1.063e+05
  * peripherals  (peripherals) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device       (peripherals) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register     (peripherals) <U2 48B '34' '34' '34' '34' '34' '34'
    localID      (peripherals) <U7 168B 'DOPort0' 'DOPort1' ... 'DOPort2'
Attributes:
    description:  Base DataArray for unified coordinates of type LED_Activati...
    source:       Constructed using construct_base_da function

...

<xarray.DataArray 'Valve_Activations_data' (Time: 521, peripherals: 6)> Size: 25kB
array([[ 0.,  0.,  0., nan, nan, nan],
       [nan, nan, nan,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.]])
Coordinates:
  * Time         (Time) float64 4kB 1.05e+05 1.05e+05 ... 1.063e+05 1.063e+05
  * peripherals  (peripherals) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device       (peripherals) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register     (peripherals) <U2 48B '34' '34' '34' '34' '34' '34'
    localID      (peripherals) <U11 264B 'SupplyPort0' ... 'SupplyPort2'
Attributes:
    description:  Base DataArray for unified coordinates of type Valve_Activa...
    source:       Constructed using construct_base_da function

device register      localID  \
Time          peripherals                                    
104990.222496 Nosepoke0    Behavior0       34  SupplyPort0   
              Nosepoke1    Behavior0       34  SupplyPort1   
              Nosepoke2    Behavior0       34  SupplyPort2   
              Nosepoke3    Behavior1       34  SupplyPort0   
              Nosepoke4    Behavior1       34  SupplyPort1   
...                              ...      ...          ...   
106297.642496 Nosepoke1    Behavior0       34  SupplyPort1   
              Nosepoke2    Behavior0       34  SupplyPort2   
              Nosepoke3    Behavior1       34  SupplyPort0   
              Nosepoke4    Behavior1       34  SupplyPort1   
              Nosepoke5    Behavior1       34  SupplyPort2   

                           Valve_Activations_data  
Time          peripherals                          
104990.222496 Nosepoke0                       0.0  
              Nosepoke1                       0.0  
              Nosepoke2                       0.0  
              Nosepoke3                       NaN  
              Nosepoke4                       NaN  
...                                           ...  
106297.642496 Nosepoke1                       0.0  
              Nosepoke2                       0.0  
              Nosepoke3                       0.0  
              Nosepoke4                       0.0  
              Nosepoke5                       0.0  

[3126 rows x 4 columns]


Reward_Release


<xarray.DataArray 'Reward_Release_data' (Time: 0, peripherals: 6)> Size: 0B
array([], shape=(0, 6), dtype=float64)
Coordinates:
  * Time         (Time) float64 0B 
  * peripherals  (peripherals) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device       (peripherals) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register     (peripherals) <U2 48B '49' '50' '51' '49' '50' '51'
    localID      (peripherals) <U16 384B 'PulseSupplyPort0' ... 'PulseSupplyP...
Attributes:
    description:  Base DataArray for unified coordinates of type Reward_Relea...
    source:       Constructed using construct_base_da function

,,device,register,localID,Reward_Release_data
Time,peripherals,,,,


In [8]:
#======= Inspect lookup_arrays
print(f"lookup_arrays keys: {list(md.lookup_arrays.keys())}\n")

for dk in md.lookup_arrays:
    lookup = md.lookup_arrays[dk]
    print(f"\n{'='*60}")
    print(f"{dk}")
    print(f"{'='*60}")
    display(lookup)
    display(lookup.to_dataframe().reset_index())

lookup_arrays keys: ['Activations', 'LED_Activations', 'Valve_Activations', 'Reward_Release']


Activations


<xarray.DataArray 'Activations_lookup' (peripherals: 6)> Size: 6B
array([ True,  True,  True,  True,  True,  True])
Coordinates:
  * peripherals  (peripherals) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device       (peripherals) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register     (peripherals) <U2 48B '32' '32' '32' '32' '32' '32'
    localID      (peripherals) <U7 168B 'DIPort0' 'DIPort1' ... 'DIPort2'
Attributes:
    description:  Lookup matrix indicating presence of unified coordinates ac...
    source:       Constructed using construct_lookup_array function

,peripherals,device,register,localID,Activations_lookup
0,Nosepoke0,Behavior0,32,DIPort0,True
1,Nosepoke1,Behavior0,32,DIPort1,True
2,Nosepoke2,Behavior0,32,DIPort2,True
3,Nosepoke3,Behavior1,32,DIPort0,True
4,Nosepoke4,Behavior1,32,DIPort1,True
5,Nosepoke5,Behavior1,32,DIPort2,True



LED_Activations


<xarray.DataArray 'LED_Activations_lookup' (peripherals: 6)> Size: 6B
array([ True,  True,  True,  True,  True,  True])
Coordinates:
  * peripherals  (peripherals) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device       (peripherals) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register     (peripherals) <U2 48B '34' '34' '34' '34' '34' '34'
    localID      (peripherals) <U7 168B 'DOPort0' 'DOPort1' ... 'DOPort2'
Attributes:
    description:  Lookup matrix indicating presence of unified coordinates ac...
    source:       Constructed using construct_lookup_array function

...

<xarray.DataArray 'Valve_Activations_lookup' (peripherals: 6)> Size: 6B
array([ True,  True,  True,  True,  True,  True])
Coordinates:
  * peripherals  (peripherals) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device       (peripherals) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register     (peripherals) <U2 48B '34' '34' '34' '34' '34' '34'
    localID      (peripherals) <U11 264B 'SupplyPort0' ... 'SupplyPort2'
Attributes:
    description:  Lookup matrix indicating presence of unified coordinates ac...
    source:       Constructed using construct_lookup_array function

,peripherals,device,register,localID,Valve_Activations_lookup
0,Nosepoke0,Behavior0,34,SupplyPort0,True
1,Nosepoke1,Behavior0,34,SupplyPort1,True
2,Nosepoke2,Behavior0,34,SupplyPort2,True
3,Nosepoke3,Behavior1,34,SupplyPort0,True
4,Nosepoke4,Behavior1,34,SupplyPort1,True
5,Nosepoke5,Behavior1,34,SupplyPort2,True



Reward_Release


<xarray.DataArray 'Reward_Release_lookup' (peripherals: 6)> Size: 6B
array([ True,  True,  True,  True,  True,  True])
Coordinates:
  * peripherals  (peripherals) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device       (peripherals) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register     (peripherals) <U2 48B '49' '50' '51' '49' '50' '51'
    localID      (peripherals) <U16 384B 'PulseSupplyPort0' ... 'PulseSupplyP...
Attributes:
    description:  Lookup matrix indicating presence of unified coordinates ac...
    source:       Constructed using construct_lookup_array function

,peripherals,device,register,localID,Reward_Release_lookup
0,Nosepoke0,Behavior0,49,PulseSupplyPort0,True
1,Nosepoke1,Behavior0,50,PulseSupplyPort1,True
2,Nosepoke2,Behavior0,51,PulseSupplyPort2,True
3,Nosepoke3,Behavior1,49,PulseSupplyPort0,True
4,Nosepoke4,Behavior1,50,PulseSupplyPort1,True
5,Nosepoke5,Behavior1,51,PulseSupplyPort2,True


## 2.3| Accessing the Collected DataFrames

Users can also call the individual pipeline steps directly. For example, <code>_collect_device_dfs()</code> returns the nested dictionary of DataFrames that the pipeline uses internally. This is the same structure as <code>get_dict_of_devices_register_dfs_dicts()</code> from the main Demonstration Workflow.

In [9]:
#======= Collect DataFrames using _collect_device_dfs
dfs_dict = md._collect_device_dfs()

print(f"Top-level keys (devices): {list(dfs_dict.keys())}")
for device in dfs_dict:
    print(f"\n  {device} registers: {list(dfs_dict[device].keys())}")

#======= Display a specific register DataFrame
print(f"\n\n{'='*60}")
print(f"Behavior0, Register 32:")
print(f"{'='*60}")
display(dfs_dict['Behavior0']['32'])


--- Processing devices and registers ---

Processing device: Behavior0
  - Loaded register TimestampSeconds (address 8) with shape (1313, 1)
  - Loaded register DigitalInputState (address 32) with shape (8, 4)
  - Loaded register OutputSet (address 34) with shape (424, 14)
  - Loaded register OutputClear (address 35) with shape (447, 14)
  - Loaded register AnalogData (address 44) with shape (1312286, 3)
  - Loaded register StartCameras (address 78) with shape (1, 2)
  - Loaded register Camera0Frame (address 92) with shape (78682, 1)

Processing device: Behavior1
  - Loaded register TimestampSeconds (address 8) with shape (1313, 1)
  - Loaded register DigitalInputState (address 32) with shape (6, 4)
  - Loaded register OutputSet (address 34) with shape (423, 14)
  - Loaded register OutputClear (address 35) with shape (447, 14)
  - Loaded register AnalogData (address 44) with shape (1312389, 3)
  - Loaded register StartCameras (address 78) with shape (1, 2)
  - Loaded register Camera0F

,DIPort0,DIPort1,DIPort2,DI3
Time,,,,
105633.255072,True,False,False,False
105633.568576,False,False,False,False
105673.389504,True,False,False,False
105673.471680,False,False,False,False
106035.302304,False,True,False,False
106035.322752,False,False,False,False
106043.345664,False,False,True,False
106043.443744,False,False,False,False


# **Part 3: Using Nosepoke**
-----

## 3.1| Nosepoke as a Convenience Wrapper

<code>Nosepoke</code> is a thin subclass of <code>MultiDevice</code> that pre-sets nosepoke-appropriate defaults so users don't have to specify them every time:
- <code>global_coord_name = 'peripherals'</code>
- <code>virtual_coord_names = ['device', 'register', 'localID']</code>
- <code>data_keys = ['Activations', 'LEDs', 'Valves', 'Rewards']</code>
- <code>device_type = 'Behavior'</code>

Users still provide their own <code>virtual_maps</code>. The default <code>data_keys</code> can be overridden if the experiment uses different type key names.

**NB: Since <code>Nosepoke</code> delegates entirely to <code>MultiDevice</code>, the outputs are identical.**

In [17]:
#======= Instantiate Nosepoke
np_obj = Nosepoke(
    virtual_maps              = virtual_maps,
    data_keys                 = type_keys,              # override default data_keys to match our type_key names
    experiment_directory_path = experiment_directory_path,
    harp_device_yaml_path     = harp_device_yaml_path,
    device_list               = device_list,
    device_registers_dict     = device_registers_dict,
    verbose                   = False,
    fill_value= False,
)

In [18]:
#======= Inspect Nosepoke outputs
for dk in np_obj.data_arrays:
    da = np_obj.data_arrays[dk]
    print(f"\n{'='*60}")
    print(f"{dk}")
    print(f"{'='*60}")
    display(da.to_dataframe().unstack('peripherals').drop(columns=['device', 'register', 'localID'], level=0))


Activations


Activations_data                                          \
peripherals          Nosepoke0 Nosepoke1 Nosepoke2 Nosepoke3 Nosepoke4   
Time                                                                     
105556.648256              0.0       0.0       0.0       0.0       0.0   
105556.684160              0.0       0.0       0.0       0.0       0.0   
105556.701728              0.0       0.0       0.0       0.0       0.0   
105556.788480              0.0       0.0       0.0       0.0       0.0   
105633.255072              1.0       0.0       0.0       0.0       0.0   
105633.568576              0.0       0.0       0.0       0.0       0.0   
105673.389504              1.0       0.0       0.0       0.0       0.0   
105673.471680              0.0       0.0       0.0       0.0       0.0   
105697.369472              0.0       0.0       0.0       0.0       1.0   
105697.573664              0.0       0.0       0.0       0.0       0.0   
106035.302304              0.0       1.0       0.0       0.0       0.0   
106035.322752              0.0       0.0       0.0       0.0       0.0   
106043.345664              0.0       0.0       1.0       0.0       0.0   
106043.443744              0.0       0.0       0.0       0.0       0.0   

                         
peripherals   Nosepoke5  
Time                     
105556.648256       1.0  
105556.684160       0.0  
105556.701728       1.0  
105556.788480       0.0  
105633.255072       0.0  
105633.568576       0.0  
105673.389504       0.0  
105673.471680       0.0  
105697.369472       0.0  
105697.573664       0.0  
106035.302304       0.0  
106035.322752       0.0  
106043.345664       0.0  
106043.443744       0.0


LED_Activations


LED_Activations_data                                          \
peripherals              Nosepoke0 Nosepoke1 Nosepoke2 Nosepoke3 Nosepoke4   
Time                                                                         
104990.222496                  0.0       0.0       0.0       0.0       0.0   
104990.223488                  0.0       0.0       0.0       0.0       0.0   
104992.237504                  0.0       0.0       0.0       0.0       0.0   
104996.252480                  0.0       0.0       0.0       0.0       0.0   
104998.269504                  0.0       0.0       0.0       0.0       0.0   
...                            ...       ...       ...       ...       ...   
106281.557504                  0.0       0.0       0.0       0.0       0.0   
106286.575488                  0.0       0.0       0.0       0.0       0.0   
106289.592480                  0.0       0.0       0.0       0.0       0.0   
106293.612480                  0.0       0.0       0.0       0.0       0.0   
106297.642496                  0.0       0.0       0.0       0.0       0.0   

                         
peripherals   Nosepoke5  
Time                     
104990.222496       0.0  
104990.223488       0.0  
104992.237504       0.0  
104996.252480       0.0  
104998.269504       0.0  
...                 ...  
106281.557504       0.0  
106286.575488       0.0  
106289.592480       0.0  
106293.612480       0.0  
106297.642496       0.0  

[521 rows x 6 columns]


Valve_Activations


Valve_Activations_data                                          \
peripherals                Nosepoke0 Nosepoke1 Nosepoke2 Nosepoke3 Nosepoke4   
Time                                                                           
104990.222496                    0.0       0.0       0.0       0.0       0.0   
104990.223488                    0.0       0.0       0.0       0.0       0.0   
104992.237504                    0.0       0.0       0.0       0.0       0.0   
104996.252480                    0.0       0.0       0.0       0.0       0.0   
104998.269504                    0.0       0.0       0.0       0.0       0.0   
...                              ...       ...       ...       ...       ...   
106281.557504                    0.0       0.0       0.0       0.0       0.0   
106286.575488                    0.0       0.0       0.0       0.0       0.0   
106289.592480                    0.0       0.0       0.0       0.0       0.0   
106293.612480                    0.0       0.0       0.0       0.0       0.0   
106297.642496                    0.0       0.0       0.0       0.0       0.0   

                         
peripherals   Nosepoke5  
Time                     
104990.222496       0.0  
104990.223488       0.0  
104992.237504       0.0  
104996.252480       0.0  
104998.269504       0.0  
...                 ...  
106281.557504       0.0  
106286.575488       0.0  
106289.592480       0.0  
106293.612480       0.0  
106297.642496       0.0  

[521 rows x 6 columns]


Reward_Release


Time


In [12]:
#======= Verify Nosepoke outputs match MultiDevice outputs
print("Comparing Nosepoke vs MultiDevice DataArrays:\n")
for dk in type_keys:
    match = md.data_arrays[dk].equals(np_obj.data_arrays[dk])
    print(f"  {dk:>20s}  match: {match}")

Comparing Nosepoke vs MultiDevice DataArrays:

           Activations  match: True
       LED_Activations  match: True
     Valve_Activations  match: True
        Reward_Release  match: True


# **Part 4: Using BehaviorNosepoke (Deprecated)**
-----

## 4.1| BehaviorNosepoke for Backwards Compatibility

<code>BehaviorNosepoke</code> is a convenience wrapper that accepts the same old-style configuration parameters used in the original <code>BehaviorNosepoke</code> class from the main Demonstration Workflow (<code>channel_list</code>, <code>type_keys</code>, <code>channel_type_localIDs</code>, <code>channel_type_registerIDs</code>). It automatically constructs <code>virtual_maps</code> internally using the device-major cycling logic, then delegates to <code>MultiDevice</code>.

This means users can switch from the old class to the new one without changing their configuration dictionaries.

In [13]:
#======= Instantiate BehaviorNosepoke (old-style config, no virtual_maps needed)
bnp = BehaviorNosepoke(
    channel_list              = channel_list,
    type_keys                 = type_keys,
    channel_type_localIDs     = channel_type_localIDs,
    channel_type_registerIDs  = channel_type_registerIDs,
    experiment_directory_path = experiment_directory_path,
    harp_device_yaml_path     = harp_device_yaml_path,
    device_list               = device_list,
    device_registers_dict     = device_registers_dict,
    verbose                   = False,
)

In [14]:
#======= Inspect BehaviorNosepoke outputs
for dk in bnp.data_arrays:
    da = bnp.data_arrays[dk]
    print(f"\n{'='*60}")
    print(f"{dk}")
    print(f"{'='*60}")
    display(da.to_dataframe())


Activations


device register  localID  Activations_data
Time          peripherals                                               
105556.648256 Nosepoke0    Behavior0       32  DIPort0               NaN
              Nosepoke1    Behavior0       32  DIPort1               NaN
              Nosepoke2    Behavior0       32  DIPort2               NaN
              Nosepoke3    Behavior1       32  DIPort0               0.0
              Nosepoke4    Behavior1       32  DIPort1               0.0
...                              ...      ...      ...               ...
106043.443744 Nosepoke1    Behavior0       32  DIPort1               0.0
              Nosepoke2    Behavior0       32  DIPort2               0.0
              Nosepoke3    Behavior1       32  DIPort0               NaN
              Nosepoke4    Behavior1       32  DIPort1               NaN
              Nosepoke5    Behavior1       32  DIPort2               NaN

[84 rows x 4 columns]


LED_Activations


device register  localID  LED_Activations_data
Time          peripherals                                                   
104990.222496 Nosepoke0    Behavior0       34  DOPort0                   0.0
              Nosepoke1    Behavior0       34  DOPort1                   0.0
              Nosepoke2    Behavior0       34  DOPort2                   0.0
              Nosepoke3    Behavior1       34  DOPort0                   NaN
              Nosepoke4    Behavior1       34  DOPort1                   NaN
...                              ...      ...      ...                   ...
106297.642496 Nosepoke1    Behavior0       34  DOPort1                   0.0
              Nosepoke2    Behavior0       34  DOPort2                   0.0
              Nosepoke3    Behavior1       34  DOPort0                   0.0
              Nosepoke4    Behavior1       34  DOPort1                   0.0
              Nosepoke5    Behavior1       34  DOPort2                   0.0

[3126 rows x 4 columns]


Valve_Activations


device register      localID  \
Time          peripherals                                    
104990.222496 Nosepoke0    Behavior0       34  SupplyPort0   
              Nosepoke1    Behavior0       34  SupplyPort1   
              Nosepoke2    Behavior0       34  SupplyPort2   
              Nosepoke3    Behavior1       34  SupplyPort0   
              Nosepoke4    Behavior1       34  SupplyPort1   
...                              ...      ...          ...   
106297.642496 Nosepoke1    Behavior0       34  SupplyPort1   
              Nosepoke2    Behavior0       34  SupplyPort2   
              Nosepoke3    Behavior1       34  SupplyPort0   
              Nosepoke4    Behavior1       34  SupplyPort1   
              Nosepoke5    Behavior1       34  SupplyPort2   

                           Valve_Activations_data  
Time          peripherals                          
104990.222496 Nosepoke0                       0.0  
              Nosepoke1                       0.0  
              Nosepoke2                       0.0  
              Nosepoke3                       NaN  
              Nosepoke4                       NaN  
...                                           ...  
106297.642496 Nosepoke1                       0.0  
              Nosepoke2                       0.0  
              Nosepoke3                       0.0  
              Nosepoke4                       0.0  
              Nosepoke5                       0.0  

[3126 rows x 4 columns]


Reward_Release


,,device,register,localID,Reward_Release_data
Time,peripherals,,,,


In [15]:
#======= Verify BehaviorNosepoke outputs match MultiDevice outputs
print("Comparing BehaviorNosepoke vs MultiDevice DataArrays:\n")
for dk in type_keys:
    match = md.data_arrays[dk].equals(bnp.data_arrays[dk])
    print(f"  {dk:>20s}  match: {match}")

#======= Confirm auto-built virtual_maps
print(f"\nBehaviorNosepoke auto-built virtual_maps keys: {list(bnp.virtual_maps.keys())}")
print(f"\n--- Activations ---")
for ch, info in bnp.virtual_maps['Activations'].items():
    print(f"  {ch:>12s}  →  {info}")

Comparing BehaviorNosepoke vs MultiDevice DataArrays:

           Activations  match: True
       LED_Activations  match: True
     Valve_Activations  match: True
        Reward_Release  match: True

BehaviorNosepoke auto-built virtual_maps keys: ['Activations', 'LED_Activations', 'Valve_Activations', 'Reward_Release']

--- Activations ---
     Nosepoke0  →  {'device': 'Behavior0', 'register': '32', 'localID': 'DIPort0'}
     Nosepoke1  →  {'device': 'Behavior0', 'register': '32', 'localID': 'DIPort1'}
     Nosepoke2  →  {'device': 'Behavior0', 'register': '32', 'localID': 'DIPort2'}
     Nosepoke3  →  {'device': 'Behavior1', 'register': '32', 'localID': 'DIPort0'}
     Nosepoke4  →  {'device': 'Behavior1', 'register': '32', 'localID': 'DIPort1'}
     Nosepoke5  →  {'device': 'Behavior1', 'register': '32', 'localID': 'DIPort2'}
